# Red Wine Quality: Explanatory Regression Analysis

**Author:** Hussein Loubani  
**Dataset:** UCI / Kaggle, Wine Quality (Red)  
**Date:** March 2026  
**Objective:** Identify which physicochemical variables are most strongly associated with perceived red wine quality using an explanatory OLS regression model.

> **Note:** This is an *explanatory* model, not a predictive one.  
> The goal is to understand which variables matter and by how much, not to minimise prediction error.  
> Causal language is deliberately avoided throughout. All results describe associations within this observational dataset.

---

## Table of Contents

1. Imports and Setup  
2. Dataset Description  
3. Initial Data Assessment  
4. Train / Hold-out Split  
5. Hypotheses  
6. Exploratory Data Analysis  
7. Variable Transformations  
8. Multicollinearity Check (VIF)  
9. Model Fitting (Full Model)  
10. Model Interpretation  
11. Reduced Model Comparison  
12. Residual Diagnostics  
13. Hold-out Evaluation  
14. Hypothesis Answers  
15. Limitations and Improvements

---
## 1. Imports and Setup

In [ ]:
# Standard library
import sys
import warnings
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make red_wine_quality importable from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

# Local project modules
from red_wine_quality.config import (
    RANDOM_SEED,
    ALPHA,
    SKEW_CANDIDATES,
    FULL_PREDICTORS,
    REDUCED_PREDICTORS,
    FULL_OLS_FORMULA,
    STEP1_OLS_FORMULA,
    STEP2_OLS_FORMULA,
    REDUCED_OLS_FORMULA,
    STEP_FORMULAS,
)
from red_wine_quality.data_processing import (
    load_data,
    assess_data,
    descriptive_stats,
    add_log_features,
    split_data,
    compute_vif,
    analyze_duplicates,
    analyze_feature_redundancy,
    outlier_summary,
)
from red_wine_quality.statistics import (
    compute_rank_correlations,
    fit_ols,
    print_fit_summary,
    coefficient_table,
    standardized_coefficients,
    evaluate_holdout,
    compare_models,
    compare_shared_coefficients,
    stepwise_summary,
    answer_hypothesis,
    jarque_bera_report,
)
from red_wine_quality.plotting import (
    apply_global_style,
    save_figure,
    plot_outlier_boxplots,
    plot_target_distribution,
    plot_univariate_distributions,
    plot_predictors_vs_target,
    plot_correlation_bar,
    plot_correlation_heatmap,
    plot_log_transform_comparison,
    plot_vif,
    plot_coefficients,
    plot_standardized_coefficients,
    plot_residual_diagnostics,
    plot_holdout_evaluation,
    plot_violin_pair,
    plot_stratification_check,
    save_all_figures,
)

warnings.filterwarnings("ignore")
apply_global_style()

DATA_PATH   = Path.cwd().parent / "data" / "raw" / "winequality-red.csv"
FIGURES_DIR = Path.cwd().parent / "reports" / "figures"

---
## 2. Dataset Description

### Source

P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009).  
*Modeling wine preferences by data mining from physicochemical properties.*  
Decision Support Systems, 47(4), 547-553. Available via Kaggle.

### Context

The dataset contains physicochemical laboratory measurements and a sensory quality score for **Portuguese *Vinho Verde* red wines**. Each row represents one wine sample tested between **2004 and 2007**. Quality was rated by **at least three sommeliers** on a scale of 0 to 10, and the final score is the **median of their evaluations**.

### Variable Glossary

| Variable | Unit | Description |
|---|---|---|
| `fixed acidity` | g/dm3 | Non-volatile acids (mainly tartaric); contribute to crispness |
| `volatile acidity` | g/dm3 | Acetic acid; at high levels produces a vinegar-like taste |
| `citric acid` | g/dm3 | Adds freshness and flavour |
| `residual sugar` | g/dm3 | Sugar remaining after fermentation |
| `chlorides` | g/dm3 | Salt content; affects taste and texture |
| `free sulfur dioxide` | mg/dm3 | Free SO2; prevents microbial growth and oxidation |
| `total sulfur dioxide` | mg/dm3 | All SO2 forms; detectable at high concentrations |
| `density` | g/cm3 | Related to alcohol and sugar content |
| `pH` | unitless | Acidity scale; most wines are 3.0 to 4.0 |
| `sulphates` | g/dm3 | Wine additive contributing to SO2 levels |
| `alcohol` | % vol | Percentage alcohol by volume |
| **`quality`** | 0 to 10 | **Target**: median sensory score from at least 3 experts |

### Key caveats

1. **No grape variety, producer, price, or vintage information** is available. Omitted-variable bias is possible.
2. **Observational data.** We cannot establish causality, only associations.
3. **Single region and style.** All wines are *Vinho Verde* from Portugal, so findings may not generalise.

---
## 3. Initial Data Assessment

This section covers the raw dataset structure, missing values, and two deliberate simplification decisions made before any analysis: removing duplicate rows and dropping `free_sulfur_dioxide`. Both decisions are justified analytically below.


In [ ]:
df = load_data(DATA_PATH)
_ = assess_data(df)

### Outlier Exploration

Before making any data cleaning decisions, the distribution of extreme values is examined using the IQR (interquartile range) method. This screens for features with substantial outlier populations that could affect model estimates. The goal here is awareness, not automatic removal — OLS regression is sensitive to high-leverage points, so understanding which features carry them informs later modelling choices.


In [ ]:
outlier_report = outlier_summary(df)

fig = plot_outlier_boxplots(df)
plt.show()

print()
print('Several features have substantial outlier populations (residual_sugar, chlorides,')
print('sulphates, total_sulfur_dioxide). These are explored further in EDA. For this')
print('explanatory analysis, outliers are retained because:')
print('  1. They represent real wines, not measurement errors.')
print('  2. Removing them would reduce an already small dataset (n=1,599).')
print('  3. Log-transforming the skewed features (Section 7) reduces their leverage.')
print('  4. Residual diagnostics (Section 12) will confirm whether any remaining')
print('     outliers distort the model fit.')


In [ ]:
df_clean = df.drop_duplicates().copy()
print(f'After removing {df.shape[0] - df_clean.shape[0]:,} duplicate rows: {df_clean.shape[0]:,} rows remain.')

In [ ]:
dup_results, safe_to_remove = analyze_duplicates(df)

### Data-driven decision: should free_sulfur_dioxide be kept or dropped?

`free_sulfur_dioxide` and `total_sulfur_dioxide` measure related but not identical concepts. Free SO2 is the unbound fraction of total SO2 — it is a mathematical subset. The question is whether free SO2 carries **independent information** about wine quality once total SO2 is already in the model.

Three tests are used to answer this data-driven question.

1. **Linear and rank-order correlation** between the two variables. Strong correlation signals redundancy.
2. **Variance Inflation Factor (VIF)** when both are included together with the other raw predictors. A VIF above 5 for either variable is a sign of harmful collinearity.
3. **Regression p-value test**: fit a small OLS model containing both variables and read the p-value of `free_sulfur_dioxide` after controlling for `total_sulfur_dioxide`. If it is non-significant, the variable adds no unique explanatory power.

The decision to keep or drop is made based on the evidence from all three tests combined.


In [ ]:
redundancy = analyze_feature_redundancy(
    df_clean,
    feature="free_sulfur_dioxide",
    related_feature="total_sulfur_dioxide",
)

In [ ]:
df_clean = df_clean.drop(columns=['free_sulfur_dioxide'])
print(f'Columns after dropping free_sulfur_dioxide: {df_clean.shape[1]}')
print(f'Remaining predictors: {[c for c in df_clean.columns if c != "quality"]}')

In [ ]:
# Save the cleaned dataset to data/interim/ for reproducibility
INTERIM_PATH = Path.cwd().parent / "data" / "interim" / "winequality_clean.csv"
df_clean.to_csv(INTERIM_PATH, index=False)
print(f"Interim dataset saved: {INTERIM_PATH} ({df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns)")


In [ ]:
desc = descriptive_stats(df_clean)
desc.style \
    .background_gradient(subset=['skew'], cmap='RdYlGn_r') \
    .background_gradient(subset=['cv'],   cmap='Blues') \
    .format('{:.3f}')

In [ ]:
fig = plot_target_distribution(df_clean)
plt.show()

print()
print('Quality is concentrated at scores 5 and 6, which together account for roughly 82% of samples.')
print('Only 10 wines were rated 3 and 18 were rated 8, making the tails very thin.')
print('OLS treats quality as continuous, which is a known simplification over ordinal regression.')

---
## 4. Train / Hold-out Split

A random 20% hold-out set is reserved before any modelling decisions are made. This prevents information from the test set from influencing variable selection, transformations, or model tuning, which would optimistically bias the hold-out evaluation.

The split uses `stratify=df["quality"]` with a fixed `random_state` because the quality distribution is heavily imbalanced. Scores 5 and 6 together account for roughly 82% of all samples, while scores 3 and 8 each represent fewer than 20 wines. A purely random split could, by chance, exclude an entire rare class from the test set or concentrate it there, distorting both the EDA statistics and the final evaluation. Stratification enforces that each quality level appears in training and test sets in approximately the same proportion as in the full dataset.


In [ ]:
train_df, test_df = split_data(df_clean)

In [ ]:
fig = plot_stratification_check(train_df, test_df)
plt.show()

print('The proportions are closely matched across all quality scores, confirming that')
print('stratification preserved the class balance between training and hold-out sets.')

---
## 5. Hypotheses

### Primary Hypothesis

Based on oenological knowledge, acetic acid at high concentrations produces vinegar-like off-flavours in wine.

| | Statement |
|---|---|
| **H1** | Higher volatile acidity is **negatively** associated with perceived wine quality, after controlling for other physicochemical variables. |
| **H0** | Volatile acidity has **no association** with wine quality once other included variables are held constant (its regression coefficient equals zero). |

### Secondary Hypothesis

| | Statement |
|---|---|
| **H1** | Higher alcohol content is **positively** associated with perceived wine quality. |
| **H0** | Alcohol content has no association with wine quality in the fitted model. |

**Significance level:** alpha = 0.05  
Both hypotheses will be formally tested in Section 14 using the OLS coefficient p-values and 95% confidence intervals.

---
## 6. Exploratory Data Analysis

All EDA below is conducted on the **training set only** to avoid any information leakage from the hold-out set.

### 6.1 Univariate Distributions

In [ ]:
predictors = [c for c in train_df.columns if c != 'quality']

fig = plot_univariate_distributions(train_df, predictors)
plt.show()

print('Right-skewed variables (absolute skewness greater than 1.0), candidates for log transformation:')
for col in predictors:
    sk = train_df[col].skew()
    if abs(sk) > 1.0:
        print(f'  {col:<30} skewness = {sk:.2f}')

### 6.2 Each Predictor vs. Quality

In [ ]:
fig = plot_predictors_vs_target(train_df, predictors)
plt.show()

### 6.3 Correlation with Quality (Ranked)

In [ ]:
rank_corr = compute_rank_correlations(train_df)

fig = plot_correlation_bar(train_df)
plt.show()

print()
print('The left panel shows Pearson r (linear correlation). The right panel shows Spearman rho')
print('(rank-order correlation), which is more appropriate for an ordinal target like quality.')
print()
print('The two measures agree closely, confirming that the linear signal is not driven by')
print('outliers or non-monotonic patterns. Alcohol and volatile acidity are the two strongest')
print('predictors on both scales.')
print()
print('Pearson r / Spearman rho comparison (training set):')
print(rank_corr.round(3).to_string())


### 6.4 Full Correlation Heatmap (Multicollinearity Screen)

In [ ]:
fig = plot_correlation_heatmap(train_df)
plt.show()

corr_matrix = train_df.corr(numeric_only=True)
print('High inter-predictor correlations (absolute r greater than 0.5):')
for col1 in predictors:
    for col2 in predictors:
        if col1 < col2:
            r = corr_matrix.loc[col1, col2]
            if abs(r) > 0.5:
                print(f'  {col1:<30} and {col2:<30} r = {r:.3f}')

### 6.5 Deep Dive: Alcohol and Volatile Acidity

In [ ]:
fig = plot_violin_pair(train_df, 'alcohol', 'volatile_acidity')
plt.show()

The violin plots reinforce the bivariate pattern. Higher quality wines tend to have higher alcohol content and lower volatile acidity, with the group means shifting monotonically across most quality levels.

---
## 7. Variable Transformations

### Rationale

OLS assumes a linear relationship between predictors and the response. For right-skewed variables, a **log1p transform** (that is, log(x + 1)) achieves two goals.

1. It **reduces skewness**, producing a more symmetric distribution so that extreme values have less leverage on the fit.
2. It **changes the coefficient interpretation** to a semi-elasticity: a 1% increase in the original variable is associated with an approximate change of beta / 100 in the predicted quality.

The log1p variant is used instead of a plain log to safely handle values near zero.

**Transformed variables:** `residual_sugar`, `chlorides`, `total_sulfur_dioxide`, `sulphates`.

In [ ]:
fig = plot_log_transform_comparison(train_df, SKEW_CANDIDATES)
plt.show()

In [ ]:
train_rdf = add_log_features(train_df)
test_rdf  = add_log_features(test_df)

print("Log features added to train and test sets.")
print([c for c in train_rdf.columns if c.startswith("log_")])

# Save model-ready splits to data/processed/
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
train_rdf.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_rdf.to_csv(PROCESSED_DIR / "test.csv",  index=False)
print(f"\nProcessed data saved to {PROCESSED_DIR}")
print(f"  train.csv : {train_rdf.shape[0]:,} rows x {train_rdf.shape[1]} columns")
print(f"  test.csv  : {test_rdf.shape[0]:,} rows x {test_rdf.shape[1]} columns")


---
## 8. Multicollinearity Check (VIF)

Variance Inflation Factor (VIF) quantifies how much the variance of a coefficient is inflated by correlation with other predictors.

| VIF range | Assessment |
|---|---|
| Below 5 | Acceptable |
| 5 to 10 | Moderate concern |
| Above 10 | High; consider removing the variable |

In an explanatory model, multicollinearity inflates standard errors and widens confidence intervals, making it harder to isolate each variable's unique contribution. The variable `density` is chemically determined by alcohol and residual sugar, and it is highly correlated with `fixed_acidity` (r close to 0.67) and `alcohol` (r close to -0.50). It will be excluded.

In [ ]:
all_features_with_density = FULL_PREDICTORS + ['density']

vif_before = compute_vif(train_rdf, all_features_with_density)
print('VIF including density:')
print(vif_before.round(2).to_string())

print()
vif_after = compute_vif(train_rdf, FULL_PREDICTORS)
print('VIF after removing density:')
print(vif_after.round(2).to_string())

In [ ]:
fig = plot_vif(vif_before, vif_after)
plt.show()

print('Density is excluded from the model. Removing it brings all VIFs below 5,')
print('which ensures that coefficient estimates and their standard errors are stable.')

---
## 9. Model Fitting (Full Model)

### Variable selection rationale

| Variable | Included | Reason |
|---|---|---|
| `volatile_acidity` | Yes | Strongest negative bivariate signal; clear oenological justification |
| `alcohol` | Yes | Strongest positive bivariate signal; well-documented quality driver |
| `log_sulphates` | Yes | Positive correlation; antimicrobial preservative effect |
| `log_chlorides` | Yes | Negative association; high salt levels are detrimental |
| `log_total_sulfur_dioxide` | Yes | High SO2 concentrations are detectable as off-flavour |
| `pH` | Yes | Acid balance indicator; modest but present signal |
| `fixed_acidity` | Yes | Controls for the acid profile alongside volatile acidity |
| `citric_acid` | Yes | Chemical completeness; avoids potential omission bias |
| `log_residual_sugar` | Yes | Low bivariate signal but chemically relevant |
| `density` | **Excluded** | High VIF; collinear with alcohol and fixed acidity |
| `free_sulfur_dioxide` | **Excluded** | Conceptually contained within total sulfur dioxide |

In [ ]:
full_model = fit_ols(train_rdf, formula=FULL_OLS_FORMULA)
print(full_model.summary())

---
## 10. Model Interpretation

### 10.1 Overall Fit

In [ ]:
print_fit_summary(full_model)

An R-squared near 0.35 is moderate and expected for this dataset. Important quality drivers such as grape variety, producer, vintage, and storage conditions are not available. The highly significant F-statistic confirms that the model explains meaningfully more variance than a null (intercept-only) baseline.

### 10.2 Coefficient Plot with 95% Confidence Intervals

In [ ]:
coef_df = coefficient_table(full_model)

fig = plot_coefficients(coef_df)
plt.show()

### 10.3 Coefficient Table

In [ ]:
display_df = coef_df[['coef', 'se', 't', 'pvalue', 'ci_low', 'ci_high', 'significant']].copy()
display_df.columns = ['Coeff', 'SE', 't', 'p-value', 'CI 2.5%', 'CI 97.5%', 'Significant']

display_df.style \
    .background_gradient(subset=['Coeff'],   cmap='RdBu', vmin=-2, vmax=2) \
    .background_gradient(subset=['p-value'], cmap='Reds_r') \
    .format({'Coeff': '{:.4f}', 'SE': '{:.4f}', 't': '{:.3f}',
             'p-value': '{:.4f}', 'CI 2.5%': '{:.4f}', 'CI 97.5%': '{:.4f}'})

### 10.4 Interpreting the Coefficients

The coefficients below are on the **raw (unstandardized) scale**. Because the predictors use different units and some are log-transformed, comparing raw coefficient magnitudes across variables is not meaningful. A coefficient of -2.5 on `log_chlorides` does not automatically indicate a larger effect than +0.28 on `alcohol`, because the scales differ.

**Untransformed predictors** have a direct interpretation. Each one-unit increase in the predictor is associated with the coefficient's value as a change in quality, holding all else constant.

| Variable | Coeff | Interpretation |
|---|---|---|
| `alcohol` | +0.28 | Each additional 1 percentage point of alcohol is associated with approximately +0.28 quality points. |
| `volatile_acidity` | -0.98 | Each additional 1 g/dm3 of acetic acid is associated with approximately one full point lower quality. |
| `pH` | -0.65 | A one-unit increase in pH (meaning less acidic) is associated with roughly 0.65 lower quality, holding other acid variables constant. |

**Log-transformed predictors** use a semi-elasticity interpretation. Because log1p(x) is approximately log(x) for values that are not very close to zero, the coefficient beta means that a 1% increase in the original variable is associated with approximately beta / 100 change in quality. A 10% increase corresponds to approximately beta multiplied by log(1.10), which equals beta multiplied by 0.0953.

| Variable | Coeff | Effect of 1% increase | Effect of 10% increase |
|---|---|---|---|
| `log_sulphates` | +1.76 | +0.018 quality points | +0.168 quality points |
| `log_chlorides` | -2.55 | -0.025 quality points | -0.243 quality points |
| `log_total_sulfur_dioxide` | -0.18 | -0.002 quality points | -0.017 quality points |
| `log_residual_sugar` | +0.01 | near zero | near zero |

**Not statistically significant (p > 0.05):** `fixed_acidity`, `citric_acid`, and `log_residual_sugar`. Their 95% confidence intervals include zero, meaning we cannot rule out that their true association with quality is zero in this dataset.

### 10.5 Standardized Coefficients (Relative Importance)

To compare the relative importance of predictors on a common scale, standardized (beta) coefficients are computed. Each standardized coefficient represents the expected change in quality, measured in standard deviations of quality, for a one standard-deviation increase in the predictor. This allows direct comparison of effect sizes across variables that use different units.

In [ ]:
std_coefs = standardized_coefficients(full_model, train_rdf)

print('Standardized (beta) coefficients, ranked by absolute magnitude:')
print(std_coefs[['raw_coef', 'std_coef']].round(4).to_string())

In [ ]:
fig = plot_standardized_coefficients(std_coefs)
plt.show()

print('On the standardized scale, alcohol has the largest positive association with quality,')
print('followed by log_sulphates. Volatile acidity has the largest negative association.')
print('These comparisons are valid because all variables have been rescaled to unit variance.')

---
## 11. Stepwise Backward Elimination

The full model contains three predictors whose p-values are well above 0.05: `log_residual_sugar`, `citric_acid`, and `fixed_acidity`. Rather than dropping all three at once, a stepwise backward approach removes them one at a time, in order of decreasing p-value. At each step the model is re-estimated and the metrics are checked before proceeding to the next removal. This makes the reasoning transparent and guards against accidentally removing a variable that becomes important once others are dropped.

Each step is evaluated on adjusted R-squared, AIC, and BIC. AIC rewards predictive fit; BIC applies a stronger penalty for model complexity. A good parsimonious model should show equal or improved BIC at each step while keeping adjusted R-squared stable.


In [ ]:
# Step 1: remove log_residual_sugar (highest p-value in full model)
step1_model = fit_ols(train_rdf, formula=STEP1_OLS_FORMULA)

print('Step 1: dropped log_residual_sugar')
print('-' * 45)
p_val = full_model.pvalues.get('log_residual_sugar', None)
if p_val is not None:
    print(f'  p-value in full model: {p_val:.4f}')
print_fit_summary(step1_model)


In [ ]:
# Step 2: remove citric_acid (highest p-value in step-1 model)
step2_model = fit_ols(train_rdf, formula=STEP2_OLS_FORMULA)

print('Step 2: dropped citric_acid')
print('-' * 45)
p_val = step1_model.pvalues.get('citric_acid', None)
if p_val is not None:
    print(f'  p-value in step-1 model: {p_val:.4f}')
print_fit_summary(step2_model)


In [ ]:
# Step 3: remove fixed_acidity (highest p-value in step-2 model)
reduced_model = fit_ols(train_rdf, formula=REDUCED_OLS_FORMULA)

print('Step 3: dropped fixed_acidity (final reduced model)')
print('-' * 45)
p_val = step2_model.pvalues.get('fixed_acidity', None)
if p_val is not None:
    print(f'  p-value in step-2 model: {p_val:.4f}')
print_fit_summary(reduced_model)


In [ ]:
models = [full_model, step1_model, step2_model, reduced_model]
labels = ['Full (9)', 'Step 1 (8)', 'Step 2 (7)', 'Reduced (6)']

step_comparison = stepwise_summary(models, labels)

print('Stepwise comparison:')
print(step_comparison.to_string())
print()
print('BIC decreases at each step, confirming that removing each non-significant variable')
print('improves the information-theoretic balance between fit and complexity.')

In [ ]:
comparison = compare_models(full_model, reduced_model, train_rdf, test_rdf)
print('Full vs Reduced Model Comparison (hold-out metrics)')
print('=' * 55)
print(comparison.round(4).to_string())
print()
print('Hold-out performance is near-identical, confirming that the three dropped variables')
print('were not capturing genuine signal beyond what the retained predictors already explain.')


In [ ]:
coef_stability = compare_shared_coefficients(full_model, reduced_model)
print('Coefficient stability for shared predictors:')
print(coef_stability.round(4).to_string())

### Model selection conclusion

At each step of the backward elimination, BIC declined while adjusted R-squared remained essentially unchanged. The coefficients of the retained predictors shifted by only small percentages at each step, confirming that the removed variables were not acting as confounders.

The hold-out performance of the reduced model is indistinguishable from that of the full model, which further supports the conclusion that the dropped terms were not contributing genuine explanatory signal.

The reduced six-predictor model is adopted as the preferred explanatory specification. It is more parsimonious, has a lower BIC, and produces slightly narrower confidence intervals for the retained coefficients. All subsequent diagnostics and hold-out evaluation use this model.


---
## 12. Residual Diagnostics

For OLS assumptions to hold, residuals should meet three conditions.

1. **Approximate normality.** The distribution of residuals should be roughly bell-shaped.
2. **Homoscedasticity.** The variance of residuals should remain roughly constant across fitted values.
3. **No systematic pattern.** Residuals should not be correlated with fitted values.

In [ ]:
fig = plot_residual_diagnostics(reduced_model)
plt.show()

print()
jarque_bera_report(reduced_model.resid)

---
## 13. Hold-out Evaluation

The reduced model is now applied to the 20% hold-out set, which was completely excluded from all modelling decisions.

In [ ]:
holdout = evaluate_holdout(reduced_model, test_rdf)

In [ ]:
fig = plot_holdout_evaluation(
    holdout.actuals, holdout.predictions,
    holdout.rmse, holdout.r2
)
plt.show()

### A note on hold-out R-squared being higher than training R-squared

The hold-out R-squared is slightly higher than the training R-squared. This is counterintuitive at first glance, because a model fitted on training data is expected to perform at least as well on that data as on unseen observations.

The explanation is sampling variability, not model superiority. The hold-out set contains only around 270 observations. The particular 20% fold drawn by the stratified random split can happen to have modestly lower residual variance than the training set purely by chance. The model did not see this fold during fitting, so the result reflects a favourable draw rather than a genuine improvement in predictive accuracy on new data.

This is a normal statistical fluctuation with small test sets and does not indicate overfitting in reverse. An overfit model would show a large drop from training to test R-squared. The near-zero gap here is strong evidence that the model is appropriately specified.


---
## 14. Hypothesis Answers

In [ ]:
print('=' * 58)
print('PRIMARY HYPOTHESIS: Volatile Acidity')
print('H1: Higher volatile acidity is negatively associated with quality.')
print('H0: No association after adjustment.')
print('=' * 58)
answer_hypothesis(reduced_model, 'volatile_acidity', expected_direction='negative')

print()
print('=' * 58)
print('SECONDARY HYPOTHESIS: Alcohol')
print('H1: Higher alcohol is positively associated with quality.')
print('H0: No association after adjustment.')
print('=' * 58)
answer_hypothesis(reduced_model, 'alcohol', expected_direction='positive')

---
## 15. Limitations and Improvements

### Limitations

1. **No causal inference.** The data is observational. We cannot claim that reducing volatile acidity would cause better quality scores. Confounding from producer skill, grape variety, or storage conditions may explain part of the observed association.

2. **Omitted variable bias.** Grape variety, producer, vintage, storage conditions, and serving temperature are absent from the dataset. These likely account for a substantial share of the unexplained variance. An R-squared near 0.35 reflects this ceiling for physicochemical predictors alone.

3. **Ordinal target treated as continuous.** Quality is a bounded integer score from 0 to 10. OLS treats it as a continuous real-valued outcome, which is a well-known simplification. Ordinal logistic regression would be the theoretically correct model, as it respects the ranked but non-metric nature of the target and handles the bounded scale without assuming equal spacing between adjacent scores. A binary framing (for example, quality >= 7 coded as "good") could also be explored as a complement to the regression findings. For this project the OLS approximation is retained because it is interpretable, widely used, and the sample is large enough that the practical difference is modest.

4. **Single dataset scope.** The findings apply to Portuguese Vinho Verde red wine tested between 2004 and 2007. They may not generalise to wines from other regions, grape varieties, or time periods.

### Potential Improvements

1. Fit an ordinal logistic regression as a robustness check and compare coefficient interpretations.
2. Test a binary classification model (good versus not-good) as a complement to the regression.
3. Add interaction terms, for example between alcohol and volatile acidity, to capture joint effects.
4. Apply robust standard errors to account for mild heteroscedasticity at the upper end of fitted values.
5. Acquire additional data on grape variety, producer, and vintage to reduce omitted variable bias.


---
## Executive Summary

Three physicochemical variables explain the most variance in expert wine quality ratings in this dataset.

1. **Alcohol** (positive association). On the standardized scale, alcohol has the largest positive effect. Each additional percentage point of alcohol is associated with roughly +0.28 quality points.

2. **Volatile acidity** (negative association). Acetic acid is associated with a vinegar-like taste. Each additional 1 g/dm3 of volatile acidity is associated with nearly a full one-point drop in quality.

3. **Sulphates** (positive association). This wine preservative is associated with higher quality, consistent with its antimicrobial role in wine stability.

The reduced model explains approximately 35% of variance in training quality ratings and generalises well on the hold-out set with no evidence of overfitting. The three predictors that were dropped (fixed acidity, citric acid, residual sugar) did not materially change the coefficients of the retained variables, supporting the more parsimonious specification.

**These are associations, not causal claims.** Controlled experiments or richer longitudinal data would be needed before making winemaking recommendations.

---
## Export Figures

Run this cell to save all visualisations to  as PNG files.


In [ ]:
save_all_figures(
    df_clean=df_clean,
    train_df=train_df,
    test_df=test_df,
    predictors=predictors,
    skew_candidates=SKEW_CANDIDATES,
    vif_before=vif_before,
    vif_after=vif_after,
    coef_df=coef_df,
    std_coefs=std_coefs,
    reduced_model=reduced_model,
    holdout_actuals=holdout.actuals,
    holdout_predictions=holdout.predictions,
    holdout_rmse=holdout.rmse,
    holdout_r2=holdout.r2,
    figures_dir=FIGURES_DIR,
)
